In [5]:
import pandas as pd

In [9]:
#load and inspect basic shape
df = pd.read_csv(
    "../data/raw/criteo/criteo-research-uplift-v2.1.csv.gz",
    compression="gzip"
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

Shape: (13979592, 16)

Columns:
Index(['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10',
       'f11', 'treatment', 'conversion', 'visit', 'exposure'],
      dtype='str')

Data types:
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


In [10]:
#check for missing values
missing_values = df.isna().sum()

print("Missing values by column:")
print(missing_values)

Missing values by column:
f0            0
f1            0
f2            0
f3            0
f4            0
f5            0
f6            0
f7            0
f8            0
f9            0
f10           0
f11           0
treatment     0
conversion    0
visit         0
exposure      0
dtype: int64


In [11]:
#check for duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 1259545


In [12]:
#check for unique values
unique_values = df.nunique()

print("Unique values by column:")
print(unique_values)

Unique values by column:
f0            2181959
f1                 60
f2            2051900
f3                552
f4                260
f5                132
f6               1645
f7             622143
f8               3743
f9               1594
f10            517372
f11               136
treatment           2
conversion          2
visit               2
exposure            2
dtype: int64


In [13]:
#combine the results for data quality checks
quality_summary = pd.DataFrame({
    "missing": missing_values,
    "unique_values": unique_values,
    "dtype": df.dtypes.astype(str)
})

quality_summary

,missing,unique_values,dtype
f0,0,2181959,float64
f1,0,60,float64
f2,0,2051900,float64
f3,0,552,float64
f4,0,260,float64
f5,0,132,float64
f6,0,1645,float64
f7,0,622143,float64
f8,0,3743,float64
f9,0,1594,float64


In [14]:
#validate the schema
expected_features = [f"f{i}" for i in range(12)]

expected_columns = expected_features + [
    "treatment",
    "visit",
    "conversion"
]

In [15]:
#check for missing or unexpected columns
missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("Missing expected columns:", missing_columns)
print("Unexpected extra columns:", extra_columns)

Missing expected columns: []
Unexpected extra columns: ['exposure']


In [16]:
#check feature dtypes
print("Feature data types:")
print(df[expected_features].dtypes)

Feature data types:
f0     float64
f1     float64
f2     float64
f3     float64
f4     float64
f5     float64
f6     float64
f7     float64
f8     float64
f9     float64
f10    float64
f11    float64
dtype: object


In [17]:
#check treatment/outcome dtypes
print("\nTreatment/outcome data types:")
print(df[["treatment", "visit", "conversion"]].dtypes)


Treatment/outcome data types:
treatment     int64
visit         int64
conversion    int64
dtype: object


In [18]:
#check binary values
for col in ["treatment", "visit", "conversion"]:
    values = sorted(df[col].dropna().unique())
    print(f"{col}: {values}")

treatment: [np.int64(0), np.int64(1)]
visit: [np.int64(0), np.int64(1)]
conversion: [np.int64(0), np.int64(1)]


In [19]:
#more explict validation of binary values
for col in ["treatment", "visit", "conversion"]:
    values = set(df[col].dropna().unique())
    is_binary = values.issubset({0, 1})
    
    print(f"{col}: binary 0/1 = {is_binary}")

treatment: binary 0/1 = True
visit: binary 0/1 = True
conversion: binary 0/1 = True


In [20]:
#understand treatment assignments
treatment_counts = df["treatment"].value_counts().sort_index()

print(treatment_counts)

treatment
0     2096937
1    11882655
Name: count, dtype: int64


In [21]:
#calculate percentage of treatment assignments
treatment_percentages = (
    df["treatment"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print(treatment_percentages)

treatment
0    14.999987
1    85.000013
Name: proportion, dtype: float64


In [22]:
#create a clean summary table for treatment assignment
treatment_summary = pd.DataFrame({
    "count": treatment_counts,
    "percentage": treatment_percentages
})

treatment_summary.index = treatment_summary.index.map({
    0: "Control",
    1: "Treatment"
})

treatment_summary

,count,percentage
treatment,,
Control,2096937,14.999987
Treatment,11882655,85.000013


In [23]:
#validate treatment assignment values
assert set(df["treatment"].unique()) == {0, 1}

print("Treatment assignment validated: 0 = Control, 1 = Treatment")

Treatment assignment validated: 0 = Control, 1 = Treatment


In [24]:
#check for missing values in binary columns
binary_columns = ["treatment", "visit", "conversion"]

print(df[binary_columns].isna().sum())

treatment     0
visit         0
conversion    0
dtype: int64


In [25]:
#check feature ranges
df[expected_features].describe().T

,count,mean,std,min,25%,50%,75%,max
f0,13979592.0,19.620297,5.377464,12.616365,12.616365,21.923413,24.436459,26.745255
f1,13979592.0,10.069977,0.104756,10.059654,10.059654,10.059654,10.059654,16.344187
f2,13979592.0,8.446582,0.299316,8.214383,8.214383,8.214383,8.723335,9.051962
f3,13979592.0,4.178923,1.336645,-8.398387,4.679882,4.679882,4.679882,4.679882
f4,13979592.0,10.338837,0.343308,10.280525,10.280525,10.280525,10.280525,21.123508
f5,13979592.0,4.028513,0.431097,-9.011892,4.115453,4.115453,4.115453,4.115453
f6,13979592.0,-4.155356,4.577914,-31.429784,-6.699321,-2.411115,0.294443,0.294443
f7,13979592.0,5.101765,1.205248,4.833815,4.833815,4.833815,4.833815,11.998401
f8,13979592.0,3.933581,0.056660,3.635107,3.910792,3.971858,3.971858,3.971858
f9,13979592.0,16.027638,7.018975,13.190056,13.190056,13.190056,13.190056,75.295017


In [26]:
#check treatment vs conversion distribution
pd.crosstab(
    df["treatment"],
    df["conversion"],
    margins=True
)

conversion,0,1,All
treatment,,,
0,2092874,4063,2096937
1,11845944,36711,11882655
All,13938818,40774,13979592


In [27]:
#add corresponding percentages to the treatment vs conversion distribution
pd.crosstab(
    df["treatment"],
    df["conversion"],
    normalize="index"
).mul(100)

conversion,0,1
treatment,,
0,99.806241,0.193759
1,99.691054,0.308946


In [28]:
#create a development sample for testing
dev_sample = df.sample(
    frac=0.01,
    random_state=42
)

print("Development sample shape:", dev_sample.shape)

Development sample shape: (139796, 16)


In [29]:
#save the development sample to a new CSV file
dev_sample.to_csv(
    "../data/raw/criteo/criteo-research-uplift-v2.1-dev.csv.gz",
    compression="gzip",
    index=False
)

In [30]:
#check if the development sample file was created successfully
import os

dev_path = "../data/raw/criteo/criteo-research-uplift-v2.1-dev.csv.gz"

print("File exists:", os.path.exists(dev_path))

File exists: True


In [31]:
#load the development sample and check its shape
df_dev = pd.read_csv(
    "../data/raw/criteo/criteo-research-uplift-v2.1-dev.csv.gz",
    compression="gzip"
)

print("Development dataset shape:", df_dev.shape)

Development dataset shape: (139796, 16)


In [32]:
# Save development sample to the planned project location

dev_sample.to_csv(
    "../data/samples/criteo_sample.csv",
    index=False
)

print(f"Development sample saved: {dev_sample.shape}")

Development sample saved: (139796, 16)


### Development Sample

A 1% development sample was created from the full Criteo uplift dataset using `pandas.DataFrame.sample()` with a fixed random seed of `42`. The resulting sample contains **139,796 rows**.

- **Sample fraction:** 1%
- **Sample size:** 139,796 rows
- **Random seed:** 42
- **Sampling method:** Uniform random sampling using `pandas.sample()`
- **Purpose:** Faster iteration during development and exploratory analysis
- **Final analysis:** All final results will be recalculated on the full **13.98M-row** dataset